# 🌊 Masterclass: Navier-Stokes, CFD, & The Future of Scientific AI
**A 2-Hour Interactive Training Course** · *revised for project release v5.5.0*

Welcome to this introduction to the Navier-Stokes Equations (NSE). In this interactive notebook we explore the history of fluid dynamics, break down the equations, and solve a simplified version in Python.

Finally, we look at a recent result in automated mathematics — **OpenAI's Lean 4-verified proof of finite-time blow-up for the Navier–Stokes equations** — and at the right way to read it physically: the proof is correct, it is a theorem about a *model*, and the interesting question is where that model stops describing a real fluid.

> *Note on earlier versions of this course:* they called the physics "Thermodynamic Censorship", said a "Dual-Scale Solver (LeanFlow)" proves reality prevents blow-ups, and said OpenAI "used 10,000 agents". All three are withdrawn — see the project's `CHANGELOG.md`.

## 🏛️ Module 1: History and Importance

### Who discovered it?
The Navier-Stokes equations were developed over several decades in the early 19th century. 
- **Claude-Louis Navier (1822):** A French engineer who first introduced the concept of viscosity (internal friction) into the equations of fluid motion.
- **George Gabriel Stokes (1845):** An Irish mathematician and physicist who refined Navier's work, providing the modern derivation based on continuum mechanics and internal stress.

### Why is it important and who uses it?
The NSE govern the motion of almost every fluid in the universe (liquids and gases). 
- **Aerospace Engineers** use them to design aircraft wings and reduce drag.
- **Meteorologists** use them to predict the weather and track hurricanes.
- **Formula 1 Teams** use them to optimize car aerodynamics.
- **Biomedical Engineers** use them to model blood flow through the human heart.

## 🧮 Module 2: Anatomy of the Equation

The Navier-Stokes equation is essentially **Newton's Second Law of Motion ($F = ma$) applied to fluids**.

$$ \rho \left( \frac{\partial \mathbf{u}}{\partial t} + \mathbf{u} \cdot \nabla \mathbf{u} \right) = -\nabla p + \mu \nabla^2 \mathbf{u} + \mathbf{f} $$

Let's break it down:
1. **$\rho \frac{\partial \mathbf{u}}{\partial t}$ (Unsteady Acceleration):** How the fluid's velocity changes over time at a specific point.
2. **$\rho (\mathbf{u} \cdot \nabla \mathbf{u})$ (Convective Acceleration):** How the fluid accelerates as it moves through space (this non-linear term drives turbulence, and is what any singularity would have to come from).
3. **$-\nabla p$ (Pressure Gradient):** Fluids move from high pressure to low pressure.
4. **$\mu \nabla^2 \mathbf{u}$ (Viscous Diffusion):** The "friction" of the fluid. It smooths gradients out. Whether it is *always* strong enough to prevent infinite speeds in 3D is exactly the open Millennium Prize question — so it is not a guarantee.
5. **$\mathbf{f}$ (External Forces):** Gravity or other external pushes.

## 🏆 Module 3: The Millennium Prize & OpenAI's Proof

### The Millennium Prize
The Clay Mathematics Institute offers $1,000,000 to anyone who can prove whether smooth solutions to the 3D NSE always exist, or whether they can "blow up" into a **singularity** (a point where velocity becomes infinite).

### OpenAI's result, read correctly
In September 2026 an OpenAI multi-agent system produced a proof, checked by the Lean 4 proof assistant, that with a carefully built smooth external force a solution *can* blow up in finite time. (Press figures such as "10,000 agents" are not stated in OpenAI's own publications.) **The proof is correct.** It is a theorem about the incompressible continuum model, which is what the Clay problem asks about; the version of the problem without a force remains open.

### Where the model stops describing a real fluid
The collapsing vortex core keeps a Reynolds number of about 1, so its size shrinks like $\sqrt{\nu t}$ and its speed grows like $\sqrt{\nu/t}$. Three assumptions of the model then fail **together**, at one length $\ell_* = \nu/c_s$ (about 0.7 nm in water, 45 nm in air):
- **Compressibility**: the Mach number $u/c_s$ passes 0.3 (onset of compressibility effects) about 7 picoseconds before the singularity in water;
- **Rarefaction**: the core becomes as small as the molecules (Knudsen number of order one);
- **Heating**: viscous heating $\Delta T \approx u^2/c_p$ reaches hundreds of kelvin (not a plasma).

In water, **cavitation** comes even earlier. None of this makes the proof wrong: it tells us which physics a real fluid would need that the model leaves out.

### Does something physical stop it? (what the project measured)
- Kinetic theory: the fluid description *ends* near the molecular scale, but a simulation found it does **not** stop a driven collapse.
- Compressibility and heat: on this route, they slow the core but do not stop it before $\ell_*$. On a different, high-Reynolds-number route, air stops speeding up at a local Mach number of about 0.7 — a limit on Mach number, not on velocity.

## 💻 Module 4 & 5: Viscosity in Action — the 1D Burgers Equation

The 1D Burgers equation $u_t + u\,u_x = \nu\,u_{xx}$ is a simplified sibling of NSE: it keeps the nonlinear convection and the viscous diffusion, but has no pressure and only one dimension.

It shows one real phenomenon clearly: **without viscosity ($\nu = 0$) a smooth wave steepens into a shock** — its *gradient* becomes infinite at the breaking time $t = 1/\max|u_0'|$ (here $t = 1$) while its *velocity* stays bounded. **With viscosity**, the gradient stays finite.

Important: this is **not** a simulation of OpenAI's construction, and it does not show that viscosity prevents 3D Navier–Stokes blow-up — in 1D, viscous Burgers is known to stay smooth, while in 3D that is the open question, and OpenAI's construction *includes* viscosity. Also note that the simple upwind scheme below adds some numerical diffusion of its own.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Grid setup
nx = 201
dx = 2 * np.pi / (nx - 1)
nt = 150
dt = 0.002
x = np.linspace(0, 2 * np.pi, nx)

# Initial Condition: A gentle wave
u_init = np.sin(x)

# 1. Inviscid Burgers (nu = 0)
# Convection steepens the wave; its gradient would become infinite at t = 1 (a shock).
u_math = u_init.copy()
math_history = [u_math.copy()]

for n in range(nt):
    un = u_math.copy()
    for i in range(1, nx - 1):
        u_math[i] = un[i] - un[i] * dt / dx * (un[i] - un[i-1])
    u_math[0] = un[0] - un[0] * dt / dx * (un[0] - un[-2])
    u_math[-1] = u_math[0]
    math_history.append(u_math.copy())

# 2. Viscous Burgers (nu > 0)
# Viscosity diffuses the steepening front, keeping the gradient finite (a 1D result).
nu = 0.1 # Physical viscosity
u_phys = u_init.copy()
phys_history = [u_phys.copy()]

for n in range(nt):
    un = u_phys.copy()
    for i in range(1, nx - 1):
        u_phys[i] = un[i] - un[i] * dt / dx * (un[i] - un[i-1]) + nu * dt / dx**2 * (un[i+1] - 2 * un[i] + un[i-1])
    u_phys[0] = un[0] - un[0] * dt / dx * (un[0] - un[-2]) + nu * dt / dx**2 * (un[1] - 2 * un[0] + un[-2])
    u_phys[-1] = u_phys[0]
    phys_history.append(u_phys.copy())

print("Simulation Complete!")

### 📊 Visualization: inviscid steepening vs viscous smoothing

The plot shows step 100, i.e. $t = 0.2$. The red line (inviscid) has begun to steepen; the cyan line (viscous) is visibly smoother and lower. The shock itself forms at $t = 1$.

*Exercise:* set `nt = 500` in the cell above and plot step 500 to watch the inviscid front approach a vertical jump while the viscous one stays smooth.

In [ ]:
plt.figure(figsize=(12, 6))
plt.style.use('dark_background')

plt.plot(x, math_history[0], 'w--', label='Initial Flow', alpha=0.5)

# Inviscid Burgers: steepening towards a shock (gradient blow-up)
plt.plot(x, math_history[100], 'r-', lw=2, label='Inviscid (nu = 0): steepening')

# Viscous Burgers: gradient kept finite
plt.plot(x, phys_history[100], 'c-', lw=3, label='Viscous (nu = 0.1): smoothed')

plt.title('1D Burgers at t = 0.2: inviscid steepening vs viscous smoothing', fontsize=14)
plt.xlabel('Space (x)')
plt.ylabel('Velocity (u)')
plt.legend(facecolor='black', fontsize=12)
plt.grid(color='#333333')
plt.show()

## 🎓 Conclusion

Congratulations on completing this training!

You now understand:
1. The components of the Navier-Stokes equations.
2. Why a correct theorem about a *model* is different from a statement about a *real fluid*.
3. Where the incompressible model stops applying for OpenAI's construction — one length, $\ell_* = \nu/c_s$ — and why that is a label for the result, not a refutation of it.
4. That regularized equations (e.g. adding extra smoothing) change the model; they are useful tools, but not a representation of what molecules do at $\ell_*$.

**Read more:** the project's paper and research notes at [xaviercallens/OpenAI-NSE-Epistemic-Audit](https://github.com/xaviercallens/OpenAI-NSE-Epistemic-Audit) (release v5.5.0, DOI [10.5281/zenodo.22696717](https://doi.org/10.5281/zenodo.22696717)).